# Notebook 02: ML Fundamentals — Preprocessing, Generalisation & Performance Evaluation

**Capstone Stage 1 | Modules 2, 5, 6, 7**  
**Dataset:** Polish Companies Bankruptcy (UCI ID 365)  
**Author:** Srini | Imperial College London — Professional Certificate in ML & AI

---

## Objectives

- Handle **missing data** and apply preprocessing pipelines (Module 2)
- Construct **train / validation / test splits** with stratification (Module 5)
- Apply **SMOTE oversampling** to address class imbalance (Module 7)
- Demonstrate the **bias-variance trade-off** on the credit dataset (Module 5)
- Implement a full **performance evaluation framework**: AUC-ROC, precision-recall, F1, confusion matrix (Modules 6, 7)
- Implement **k-fold cross-validation** with stratification (Module 7)

**Governance note:** Every preprocessing decision here is a model risk item. Choices made in this notebook (imputation strategy, SMOTE parameters, split proportions) are documented as model assumptions subject to independent challenge under SR 11-7.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
COLOURS = {'bankrupt': '#d62728', 'solvent': '#1f77b4'}
np.random.seed(42)

# Load data
df = pd.read_csv('../data/polish_bankruptcy.csv')
with open('../data/feature_map.json') as f:
    feature_map = json.load(f)
feature_cols = [c for c in df.columns if c.startswith('X')]
X_raw = df[feature_cols].values
y = df['target'].values
print('Data loaded:', X_raw.shape, '| Class balance:', np.bincount(y.astype(int)))

---
## 1. Preprocessing Pipeline — Imputation & Scaling

In [ ]:
# Stratified train / validation / test split — 60/20/20
# Stratification preserves the 5% bankruptcy rate in each split
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_raw, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval)

print('Split summary (stratified):')
for name, y_split in [('Train', y_train), ('Validation', y_val), ('Test', y_test)]:
    print(f'  {name:12s}: {len(y_split):5,} obs | {y_split.mean()*100:.1f}% bankrupt')

# Build preprocessing pipeline
# Choice of RobustScaler: financial ratios contain extreme outliers;
# RobustScaler uses IQR rather than std, reducing sensitivity to extremes
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # Median imputation: robust to outliers
    ('scaler', RobustScaler())                        # IQR-based scaling
])

# Fit on training data only — no data leakage from val/test
X_train_pp = preprocessor.fit_transform(X_train)
X_val_pp   = preprocessor.transform(X_val)
X_test_pp  = preprocessor.transform(X_test)

print('\nPreprocessing pipeline applied.')
print('Imputation strategy : median (robust to skewed financial distributions)')
print('Scaling strategy    : RobustScaler (IQR-based, resistant to outliers)')
print('Fit on              : training set only (prevents data leakage)')

---
## 2. SMOTE — Oversampling the Minority Class (Module 7)

In [ ]:
# SMOTE applied ONLY to training data — never to validation or test sets
# This is a critical model risk constraint: synthetic samples must not contaminate evaluation
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_pp, y_train)

print('SMOTE oversampling results:')
print(f'  Before SMOTE: {np.bincount(y_train.astype(int))} (solvent/bankrupt)')
print(f'  After SMOTE : {np.bincount(y_train_smote.astype(int))} (solvent/bankrupt)')
print(f'  Synthetic samples added: {y_train_smote.sum() - y_train.sum():,}')
print('\nGovernance note: SMOTE applied to training fold only.')
print('Validation and test sets retain original class distribution for unbiased evaluation.')

---
## 3. Baseline Performance — Naïve Classifier (Module 6)

In [ ]:
# Always-negative baseline: exposes accuracy trap
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_smote, y_train_smote)
y_pred_dummy = dummy.predict(X_val_pp)

print('Naïve Baseline (always predicts majority class):')
print(f'  Accuracy  : {(y_pred_dummy == y_val).mean()*100:.1f}%')
print(f'  F1 (bankrupt): {f1_score(y_val, y_pred_dummy, pos_label=1, zero_division=0):.3f}')
print(f'  AUC-ROC   : 0.500 (by definition — no discrimination)')
print('\nKey point: 95% accuracy is achievable by predicting nobody goes bankrupt.')
print('This benchmark makes accuracy useless as a credit model metric.')

---
## 4. Bias-Variance Trade-off — Learning Curves (Module 5)

In [ ]:
# Decision tree at different depths — classic bias-variance illustration
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

depths = [2, 6, None]  # shallow (high bias), medium, deep (high variance)
depth_labels = ['Depth=2 (High Bias)', 'Depth=6 (Balanced)', 'Depth=None (High Variance)']

for ax, depth, label in zip(axes, depths, depth_labels):
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_train_smote, y_train_smote,
        cv=StratifiedKFold(5), scoring='roc_auc',
        train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)

    ax.plot(train_sizes, train_mean, 'o-', color='#2196F3', label='Training AUC')
    ax.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.15, color='#2196F3')
    ax.plot(train_sizes, val_mean, 's-', color='#F44336', label='CV Validation AUC')
    ax.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.15, color='#F44336')
    ax.set_title(label, fontweight='bold', fontsize=10)
    ax.set_xlabel('Training examples')
    ax.set_ylabel('AUC-ROC' if ax == axes[0] else '')
    ax.legend(fontsize=8)
    ax.set_ylim(0.45, 1.02)
    ax.axhline(0.5, color='grey', linestyle=':', alpha=0.5)

plt.suptitle('Bias-Variance Trade-off — Learning Curves on Polish Bankruptcy Dataset',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/02_bias_variance_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Observation: Depth=2 shows high bias (low training AND validation AUC).')
print('Depth=None shows overfitting gap (training AUC >> validation AUC).')
print('Optimal depth ~6 balances the trade-off for this dataset.')

---
## 5. Full Performance Evaluation Framework (Modules 6 & 7)

In [ ]:
# Logistic Regression as evaluation reference model
lr = LogisticRegression(max_iter=500, random_state=42, class_weight='balanced')
lr.fit(X_train_smote, y_train_smote)
y_val_proba = lr.predict_proba(X_val_pp)[:, 1]
y_val_pred  = (y_val_proba >= 0.5).astype(int)

# AUC-ROC
auc = roc_auc_score(y_val, y_val_proba)
fpr, tpr, thresholds_roc = roc_curve(y_val, y_val_proba)

# Precision-Recall
ap = average_precision_score(y_val, y_val_proba)
prec, rec, thresholds_pr = precision_recall_curve(y_val, y_val_proba)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ROC curve
axes[0].plot(fpr, tpr, color='#2196F3', linewidth=2, label=f'LR (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1],'k--', alpha=0.4, label='Random (AUC=0.500)')
axes[0].fill_between(fpr, tpr, alpha=0.08, color='#2196F3')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontweight='bold'); axes[0].legend()

# Precision-Recall curve
baseline_pr = y_val.mean()
axes[1].plot(rec, prec, color='#F44336', linewidth=2, label=f'LR (AP={ap:.3f})')
axes[1].axhline(baseline_pr, color='grey', linestyle='--', alpha=0.6, label=f'Baseline ({baseline_pr:.2f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold'); axes[1].legend()

# Confusion matrix
cm = confusion_matrix(y_val, y_val_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Solvent', 'Bankrupt'])
disp.plot(ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title('Confusion Matrix (threshold=0.5)', fontweight='bold')

plt.suptitle('Performance Evaluation Framework — Logistic Regression Baseline',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/02_performance_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(y_val, y_val_pred, target_names=['Solvent', 'Bankrupt']))
print(f'AUC-ROC           : {auc:.4f}')
print(f'Average Precision  : {ap:.4f}')

---
## 6. Stratified K-Fold Cross-Validation (Module 7)

In [ ]:
# 5-fold stratified cross-validation on training data
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Run CV on pre-SMOTE data with pipeline including SMOTE per fold
from imblearn.pipeline import Pipeline as ImbPipeline

cv_pipeline = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('smote', SMOTE(random_state=42, k_neighbors=5)),
    ('classifier', LogisticRegression(max_iter=500, random_state=42, class_weight='balanced'))
])

cv_auc_scores = cross_val_score(cv_pipeline, X_train, y_train, cv=skf, scoring='roc_auc', n_jobs=-1)
cv_f1_scores  = cross_val_score(cv_pipeline, X_train, y_train, cv=skf, scoring='f1', n_jobs=-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, scores, metric in [(axes[0], cv_auc_scores, 'AUC-ROC'), (axes[1], cv_f1_scores, 'F1 (Bankrupt)')]:
    ax.bar(range(1, 6), scores, color='#2196F3', alpha=0.8, edgecolor='navy')
    ax.axhline(scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean={scores.mean():.3f}')
    ax.fill_between([-0.5, 5.5],
                    scores.mean()-scores.std(), scores.mean()+scores.std(),
                    alpha=0.15, color='red', label=f'±1 SD ({scores.std():.3f})')
    ax.set_xlabel('Fold'); ax.set_ylabel(metric)
    ax.set_title(f'5-Fold CV — {metric}', fontweight='bold')
    ax.legend(); ax.set_xlim(0.3, 5.7)

plt.suptitle('Stratified K-Fold Cross-Validation — Logistic Regression\n(SMOTE applied within each fold)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/02_kfold_cv.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'CV AUC-ROC: {cv_auc_scores.mean():.4f} ± {cv_auc_scores.std():.4f}')
print(f'CV F1     : {cv_f1_scores.mean():.4f} ± {cv_f1_scores.std():.4f}')
print('\nLow standard deviation across folds indicates stable generalisation.')

---
## 7. Summary & Saved Artefacts

The preprocessing pipeline and train/test splits built in this notebook are the foundation for all subsequent modelling notebooks. Key decisions documented:

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Missing imputation | Median | Robust to skewed financial distributions |
| Scaling | RobustScaler | IQR-based, resistant to outlier financial ratios |
| Class imbalance | SMOTE (k=5) | Synthetic minority oversampling within training fold only |
| Split | 60/20/20 stratified | Preserves 4.9% bankruptcy rate across all splits |
| CV strategy | 5-fold stratified | Stable estimates on imbalanced dataset |
| Primary metric | AUC-ROC + F1 | Accuracy misleading at 5% prevalence |

---
## Next Notebook

→ **Notebook 03:** KNN, Decision Trees & Ensemble Methods (Modules 8, 9, 10)